In [ ]:
import sqlite3
import json
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from collections import Counter

DB_PATH = Path("../dataset_network_final/events_sort.sqlite")
OUTPUT = Path("../dataset_network_final/fragment_final.jsonl")

assert DB_PATH.exists(), f"Database non trovato: {DB_PATH}"

def iso(ts):
    if ts is None:
        return None
    return datetime.fromtimestamp(
        ts, tz=timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

def get_categories(value):
    if value is None:
        return []
    try:
        result = json.loads(value)
        return result if isinstance(result, list) else [result]
    except (ValueError, TypeError):
        return [value]

def get_price_ton(value, is_purchase):
    if not is_purchase or value is None:
        return None
    try:
        price = Decimal(str(value)) / Decimal("1000000000")
        return float(price) if price.is_finite() else None
    except (InvalidOperation, ValueError, TypeError):
        return None

In [2]:
conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    layer, nft, asset_name, categories,
    timestamp, start_lt, end_lt,
    action_id, trace_id, action_type,
    old_owner, real_old_owner, new_owner,
    price_raw, is_purchase
FROM events
ORDER BY
    layer, nft, timestamp,
    start_lt, end_lt, action_id, rowid
"""

stats = Counter()

current_nft = None
current_owner = None
current_acquired_at = None

with OUTPUT.open("w", encoding="utf-8") as fout:

    for (
        layer, nft, asset_name, categories,
        ts, start_lt, end_lt,
        action_id, trace_id, action_type,
        old_owner, real_old_owner, new_owner,
        price_raw, is_purchase
    ) in conn.execute(query):

        nft_key = (layer, nft)

        # Nuovo NFT: azzeriamo la cronologia
        if nft_key != current_nft:
            current_nft = nft_key
            current_owner = None
            current_acquired_at = None

        purchase = (
            action_type == "nft_purchase"
            or is_purchase == 1
        )

        # Ricostruzione sender
        if current_owner is not None:
            sender = current_owner
            sender_acquired_at = current_acquired_at
        else:
            sender = old_owner or real_old_owner
            sender_acquired_at = None

        recipient = new_owner

        # Per gli eventi con destinatario mancante,
        # non interrompiamo la sequenza dei proprietari.
        if recipient is None:
            stats["NULL_RECIPIENT"] += 1

        # Stesso proprietario: il possesso continua
        elif recipient == current_owner:
            stats["SAME_OWNER"] += 1

        # Cambio di proprietario
        else:
            stats["OWNER_CHANGE"] += 1

        # La cessione del sender avviene solo quando
        # abbiamo un nuovo destinatario diverso.
        is_change = (
            sender is not None
            and recipient is not None
            and sender != recipient
        )

        sender_released_at = iso(ts) if is_change else None

        holding_days = None

        if (
            is_change
            and sender_acquired_at is not None
        ):
            holding_days = round(
                (ts - sender_acquired_at) / 86400,
                6
            )

        # Un destinatario nuovo acquisisce l'NFT al timestamp
        recipient_acquired_at = (
            iso(ts)
            if recipient is not None and recipient != current_owner
            else iso(current_acquired_at)
            if recipient is not None
            else None
        )

        record = {
            "layer": layer,
            "nft": nft,
            "asset_name": asset_name,
            "categories": get_categories(categories),

            "sender": sender,
            "recipient": recipient,

            "timestamp": iso(ts),
            "sender_acquired_at": iso(sender_acquired_at),
            "sender_released_at": sender_released_at,
            "recipient_acquired_at": recipient_acquired_at,
            "holding_days_sender": holding_days,

            "action_type": action_type,
            "is_purchase": purchase,
            "price_ton": get_price_ton(price_raw, purchase),

            "action_id": action_id,
            "trace_id": trace_id
        }

        fout.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

        stats["TOTAL"] += 1

        # Il nuovo destinatario diventa il proprietario
        # da utilizzare nell'operazione successiva.
        if (
            recipient is not None
            and recipient != current_owner
        ):
            current_owner = recipient
            current_acquired_at = ts

conn.close()

print("DATASET FINALE GENERATO")
print("File:", OUTPUT)

for key, value in stats.items():
    print(f"{key}: {value:,}")

print(
    "Dimensione:",
    round(OUTPUT.stat().st_size / 1024**3, 3),
    "GB"
)

assert stats["TOTAL"] == 3_406_889

DATASET FINALE GENERATO
File: dataset_network_final/fragment_final.jsonl
OWNER_CHANGE: 3,270,964
TOTAL: 3,406,889
SAME_OWNER: 135,925
Dimensione: 2.239 GB


In [3]:
import pandas as pd
import json
from collections import Counter

sample = pd.read_json(
    OUTPUT,
    lines=True,
    nrows=20
)

display(sample)

check = Counter()
null_holding = 0

with OUTPUT.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)

        check["totale"] += 1

        if r["sender"] is not None and r["recipient"] is not None:
            if r["sender"] != r["recipient"]:
                check["archi_wallet"] += 1

        if r["holding_days_sender"] is not None:
            check["holding_calcolati"] += 1
        else:
            null_holding += 1

        if (
            r["holding_days_sender"] is not None
            and r["holding_days_sender"] < 0
        ):
            check["holding_negativi"] += 1

print(dict(check))
print("Holding time non calcolabili:", f"{null_holding:,}")

,layer,nft,asset_name,categories,sender,recipient,timestamp,sender_acquired_at,sender_released_at,recipient_acquired_at,holding_days_sender,action_type,is_purchase,price_ton,action_id,trace_id
0,phone_numbers,0:0000adc15ce01828a10f3518ab6674393279eef39399...,+888 0645 3295,[No_Categories],None,0:07dee4afa8d8528d0f4557b9be55b8e45a7cafce790d...,2022-12-14 06:45:38+00:00,NaT,NaT,2022-12-14 06:45:38+00:00,NaN,nft_transfer,False,NaN,nfS4IwbTNsc0osCMhGvkFRoiP+4Qxni0sUOtCi3WvOE=,8dN3fxjcp2O8VQZ0Zbd3Yo1ffqv3njqTgxZY6fS5Fdc=
1,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],None,0:994116b79b56e76fc09fe8ce2a317b6e49e334aeed78...,2022-12-13 17:15:08+00:00,NaT,NaT,2022-12-13 17:15:08+00:00,NaN,nft_transfer,False,NaN,BvK+6VS6kmtLZAlgTNTtVgQycRgzL+Ywvtjat4jfcjQ=,FyIPbqBvl6p3efHNZFy8ejrdvHO2SubyxlghygLyZLM=
2,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:994116b79b56e76fc09fe8ce2a317b6e49e334aeed78...,0:3abc098fb32f3ab2288eeaf23e638c097b0e8bfa564f...,2024-01-19 01:32:18+00:00,2022-12-13 17:15:08+00:00,2024-01-19 01:32:18+00:00,2024-01-19 01:32:18+00:00,401.345255,nft_transfer,True,160.0,HVVOIXBNnEuTPxbgd+g8BJo6MhqShUHKct8UWRQfMZ8=,xmnWNoOVR23uusj6UTGNfOh+FNn2ijKPqhgNiC6hdM0=
3,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:3abc098fb32f3ab2288eeaf23e638c097b0e8bfa564f...,0:625e3abcbe5a3097b71d039654c25ebaf8a138b0c348...,2024-11-27 16:30:47+00:00,2024-01-19 01:32:18+00:00,2024-11-27 16:30:47+00:00,2024-11-27 16:30:47+00:00,313.623947,nft_transfer,False,NaN,IiyKDub54hyGUCjj38vRN9wsrVGZItpb8KhCG0jrQQg=,AZbVEHbzf6X/Tltgcbaw5GERLo1uA5X6TVJ9tk9YXxY=
4,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:625e3abcbe5a3097b71d039654c25ebaf8a138b0c348...,0:85bd48cb46bf23431adca9535e3b455a4e3eb6bd768d...,2025-01-12 04:15:37+00:00,2024-11-27 16:30:47+00:00,2025-01-12 04:15:37+00:00,2025-01-12 04:15:37+00:00,45.489468,nft_transfer,False,NaN,aAyietGLlvDOZbif/0xuKRVJ86MrUjij92DCEGBhvi4=,fDTaantWgONB3v7bqYhzt/A7JbsttsZJ5qUOO5mopRg=
5,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:85bd48cb46bf23431adca9535e3b455a4e3eb6bd768d...,0:625e3abcbe5a3097b71d039654c25ebaf8a138b0c348...,2025-01-12 04:50:21+00:00,2025-01-12 04:15:37+00:00,2025-01-12 04:50:21+00:00,2025-01-12 04:50:21+00:00,0.024120,nft_transfer,False,NaN,sgMO4/0FE1djcZH1/KMQPLaKXdpavadeHxHv1FemGY0=,UvQHUbWJXRlU7MHdRzOTz1hULpMG6Yjdo1iU8EHJSg8=
6,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:625e3abcbe5a3097b71d039654c25ebaf8a138b0c348...,0:85bd48cb46bf23431adca9535e3b455a4e3eb6bd768d...,2025-01-12 04:52:16+00:00,2025-01-12 04:50:21+00:00,2025-01-12 04:52:16+00:00,2025-01-12 04:52:16+00:00,0.001331,nft_transfer,False,NaN,U6MfHpUq/j6l4uaM5e4KMMOs28uuMBbesV2htFmegk8=,Dz8AbjiazJEa+qT2TCdKp9KAYmkmyTqq+CDU+i+zS80=
7,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:85bd48cb46bf23431adca9535e3b455a4e3eb6bd768d...,0:866370617e7b3a3e254b6ea4d7d8f95ca68eb402ca93...,2025-10-18 01:01:13+00:00,2025-01-12 04:52:16+00:00,2025-10-18 01:01:13+00:00,2025-10-18 01:01:13+00:00,278.839549,nft_transfer,False,NaN,Eqv1Fdl+/MAU7yZ86pcWnmmc4/ERT5woLOJsb2KcvyE=,UDUdKCuGyVG+tQu4rT4L8MnvDsnBnr8kfUOo14nDJd0=
8,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:866370617e7b3a3e254b6ea4d7d8f95ca68eb402ca93...,0:a9e6f94588ee3541e97bd7dce64d4cd896509085bd55...,2025-10-18 04:25:15+00:00,2025-10-18 01:01:13+00:00,2025-10-18 04:25:15+00:00,2025-10-18 04:25:15+00:00,0.141690,nft_transfer,False,NaN,CZ76l98zyq5WK1Mh7sYbdIQ5A56LchYGompe5X5rj0Y=,19zVGQtmovIr8Z1LSJTck/gj6IR4fsfEK79pVdOMZY4=
9,phone_numbers,0:0000f777522e708986ee14b710f7cdc0ebef589763bc...,+888 0694 1320,[69 Club],0:a9e6f94588ee3541e97bd7dce64d4cd896509085bd55...,0:81b14296efc54ff8df7ba44e2cfd59a4b980756f5e24...,2025-10-20 23:25:15+00:00,2025-10-18 

{'totale': 3406889, 'archi_wallet': 2579191, 'holding_calcolati': 2579191}
Holding time non calcolabili: 827,698


In [6]:
import json
from itertools import islice
from pathlib import Path

path = Path("dataset_network_final/fragment_final.jsonl")

with path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(islice(f, 20), start=1):
        record = json.loads(line)

        print(f"\n{'=' * 80}")
        print(f"RIGA {i}")
        print(f"{'=' * 80}")

        print(json.dumps(
            record,
            indent=4,
            ensure_ascii=False
        ))


RIGA 1
{
    "layer": "phone_numbers",
    "nft": "0:0000adc15ce01828a10f3518ab6674393279eef3939934fc2293f8227a629e3a",
    "asset_name": "+888 0645 3295",
    "categories": [
        "No_Categories"
    ],
    "sender": null,
    "recipient": "0:07dee4afa8d8528d0f4557b9be55b8e45a7cafce790d7aad823b34c75ecbf2c9",
    "timestamp": "2022-12-14T06:45:38Z",
    "sender_acquired_at": null,
    "sender_released_at": null,
    "recipient_acquired_at": "2022-12-14T06:45:38Z",
    "holding_days_sender": null,
    "action_type": "nft_transfer",
    "is_purchase": false,
    "price_ton": null,
    "action_id": "nfS4IwbTNsc0osCMhGvkFRoiP+4Qxni0sUOtCi3WvOE=",
    "trace_id": "8dN3fxjcp2O8VQZ0Zbd3Yo1ffqv3njqTgxZY6fS5Fdc="
}

RIGA 2
{
    "layer": "phone_numbers",
    "nft": "0:0000f777522e708986ee14b710f7cdc0ebef589763bcccee53def786ed218fb6",
    "asset_name": "+888 0694 1320",
    "categories": [
        "69 Club"
    ],
    "sender": null,
    "recipient": "0:994116b79b56e76fc09fe8ce2a317b6e49e334